# Small Reasoning LLM Lab — Training

**How to use:**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. When prompted, allow Google Drive access
4. Wait ~10-30 minutes

**All checkpoints are saved to your Google Drive automatically.**  
They survive Colab session resets. You can reload them any time.

Save location: `My Drive/small-llm-lab/experiments/results/colab_small_baseline/`

## Step 1 — Mount Google Drive (checkpoints saved here permanently)

In [ ]:
import os, sys, subprocess

# Mount Drive — all checkpoints will be saved here so they survive reboots
from google.colab import drive
drive.mount('/content/drive')

# Create the project folder on Drive
DRIVE_ROOT = '/content/drive/MyDrive/small-llm-lab'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive folder: {DRIVE_ROOT}')

# Clone or update the repo code (code lives in /content, not Drive)
REPO_URL  = 'https://github.com/sinor77/small-llm-lab.git'
REPO_NAME = 'small-llm-lab'
CODE_DIR  = f'/content/{REPO_NAME}'

def _git(*args, cwd=None):
    r = subprocess.run(['git'] + list(args), cwd=cwd or os.getcwd(),
                       capture_output=True, text=True)
    for line in (r.stdout + r.stderr).strip().splitlines():
        print(f'  git: {line}')

if not os.path.exists(CODE_DIR):
    print(f'Cloning {REPO_URL} ...')
    _git('clone', REPO_URL, cwd='/content')
else:
    print('Repo found. Pulling latest code...')
    _git('pull', 'origin', 'main', cwd=CODE_DIR)

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print(f'\nCode directory: {os.getcwd()}')
_git('log', '--oneline', '-3', cwd=CODE_DIR)

## Step 2 — Install dependencies

In [ ]:
try:
    import tokenizers
    print(f'tokenizers {tokenizers.__version__} already installed.')
except ImportError:
    print('Installing tokenizers...')
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'tokenizers>=0.15.0', 'tqdm', 'pyyaml', '-q'], check=True)
    print('Done.')

## Step 3 — Detect GPU

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'CUDA:   {torch.version.cuda}')
else:
    DEVICE = 'cpu'
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU')

print(f'PyTorch: {torch.__version__}')

## Step 4 — Hardware benchmark (measures actual GPU speed)

In [ ]:
import time
from model.config import get_8m_config
from model.transformer import SmallTransformer

cfg = get_8m_config()
cfg.vocab_size = 4096
cfg.max_seq_len = 256
bm = SmallTransformer(cfg).to(DEVICE)
bm.train()
bm_opt = torch.optim.AdamW(bm.parameters(), lr=3e-4)
xb = torch.randint(1, 4096, (64, 200), device=DEVICE)
yb = torch.randint(1, 4096, (64, 200), device=DEVICE)

for _ in range(5):  # warmup
    l, _ = bm(xb, yb); bm_opt.zero_grad(); l.backward(); bm_opt.step()
if DEVICE == 'cuda': torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(30):
    l, _ = bm(xb, yb); bm_opt.zero_grad(); l.backward(); bm_opt.step()
if DEVICE == 'cuda': torch.cuda.synchronize()
t1 = time.perf_counter()

sps = 30 / (t1 - t0)
tps = sps * 64 * 200
est_min = (20_000 * 80 * 30) / tps / 60
peak = torch.cuda.max_memory_allocated()/1e9 if DEVICE=='cuda' else 0

print(f'Steps/sec:       {sps:.1f}')
print(f'Tokens/sec:      {tps:,.0f}')
print(f'Peak GPU memory: {peak:.2f} GB')
print(f'Estimated training time (colab_small): ~{est_min:.0f} minutes')
print('(This is measured on your actual GPU — not a guess)')

del bm, bm_opt, xb, yb
if DEVICE == 'cuda': torch.cuda.empty_cache()

## Step 5 — Configure experiment (output goes to Google Drive)

In [ ]:
from training.config import get_train_config_by_name

PRESET = 'colab_small'   # debug | colab_small | colab_medium
train_config = get_train_config_by_name(PRESET)

# ── Redirect ALL output to Google Drive ──────────────────────────────────────
# This means checkpoints, tokenizer, metrics, and results all go to Drive.
# They will survive Colab session resets.
DRIVE_EXP_DIR = os.path.join(DRIVE_ROOT, train_config.output_dir)
os.makedirs(DRIVE_EXP_DIR, exist_ok=True)

train_config.output_dir       = DRIVE_EXP_DIR
train_config.checkpoint_dir   = os.path.join(DRIVE_EXP_DIR, 'checkpoints')
train_config.tokenizer_dir    = os.path.join(DRIVE_EXP_DIR, 'tokenizer')
train_config.metrics_file     = os.path.join(DRIVE_EXP_DIR, 'metrics.jsonl')
train_config.model_config_file= os.path.join(DRIVE_EXP_DIR, 'model_config.json')
train_config.train_config_file= os.path.join(DRIVE_EXP_DIR, 'train_config.json')
train_config.eval_file        = os.path.join(DRIVE_EXP_DIR, 'evaluation.json')
train_config.samples_file     = os.path.join(DRIVE_EXP_DIR, 'samples.json')

# Resume from Drive if a checkpoint already exists there
best_on_drive = os.path.join(train_config.checkpoint_dir, 'best.pt')
if os.path.exists(best_on_drive):
    print(f'Existing checkpoint found on Drive: {best_on_drive}')
    print('Training will resume from this checkpoint.')
    train_config.resume_from = best_on_drive
else:
    print('No existing checkpoint — will train from scratch.')

print(f'\nExperiment:     {train_config.experiment_name}')
print(f'Output (Drive): {DRIVE_EXP_DIR}')
print(f'N train:        {train_config.n_train:,}')
print(f'Epochs:         {train_config.max_epochs}')

## Step 6 — Generate dataset

In [ ]:
from data.generators.arithmetic import ArithmeticGenerator

generator = ArithmeticGenerator(seed=train_config.data_seed,
                                 difficulty=train_config.difficulty)

train_examples, val_examples, test_examples = generator.generate_all_splits(
    n_train=train_config.n_train,
    n_val=train_config.n_val,
    n_test=train_config.n_test,
)

tp = set(e.problem for e in train_examples)
vp = set(e.problem for e in val_examples)
sp = set(e.problem for e in test_examples)
assert len(tp & vp) == 0 and len(tp & sp) == 0 and len(vp & sp) == 0

print(f'Train: {len(train_examples):,}  Val: {len(val_examples):,}  Test: {len(test_examples):,}')
print('Zero cross-split overlap: verified')
print()
print(train_examples[0].to_reasoning_text())

## Step 7 — Train (checkpoints auto-saved to Google Drive)

In [ ]:
from training.train import train

# Checkpoints are written directly to Google Drive.
# If the session disconnects, re-run from Step 1.
# The config already has resume_from set if a checkpoint exists on Drive.

print(f'Checkpoints will be saved to:')
print(f'  {train_config.checkpoint_dir}')
print()

results = train(train_config)

print('\n=== Training complete ===')
for k, v in results.items():
    if not isinstance(v, dict):
        print(f'  {k}: {v}')

## Step 8 — Evaluate

In [ ]:
import json
from model.config import ModelConfig
from model.transformer import SmallTransformer
from model.tokenizer import BPETokenizer
from evaluation.benchmark import run_benchmark, run_generalization_benchmark

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ck_path = os.path.join(train_config.checkpoint_dir, 'best.pt')
ck      = torch.load(ck_path, map_location=device)

model     = SmallTransformer(ModelConfig.from_dict(ck['model_config']))
model.load_state_dict(ck['model_state'])
model.to(device).eval()
tokenizer = BPETokenizer.load(train_config.tokenizer_dir)

print(f'Loaded best.pt  step={ck["global_step"]}  params={model.count_parameters():,}')

bench = run_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_problems=500, split='test', max_new_tokens=64, temperature=0.0,
    use_reasoning=train_config.use_reasoning, max_samples_to_save=20, device=device,
)
print('\n' + bench.summary_str())

In [ ]:
all_known = set(e.problem for e in train_examples + val_examples + test_examples)
gen_results = run_generalization_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_per_level=200, device=device,
    use_reasoning=train_config.use_reasoning,
    exclude_problems=all_known,
)
print('Generalization (5 levels):')
for level, r in gen_results.items():
    print(f'  Level {level}: {r["accuracy"]:.1%}  ({r["correct"]}/{r["total"]})')

## Step 9 — Interactive inference

In [ ]:
from inference.generate import interactive_inference

# Type math problems and press Enter. Type 'exit' to stop.
interactive_inference(
    checkpoint_path=ck_path,
    device=device,
    max_new_tokens=128,
    temperature=0.0,
    verify_answers=True,
)

## Notes

- All files are saved to **Google Drive** at `My Drive/small-llm-lab/experiments/results/`
- If the session disconnects, re-run from Step 1. Training resumes automatically from the last checkpoint.
- To evaluate an already-trained model, open `colab/evaluate_trained_model.ipynb` instead.
- The verifier is programmatic — no LLM is used as a judge.
- Low training loss does NOT mean the model solves problems correctly. Check test accuracy.